# 논문 학습·평가 절차를 참고한 RandomForest 기준 모델

13개 컬럼을 유지하고, 추가 튜닝 없이 RandomForest의 기준 성능을 확인한다.

- 논문에서 확인된 방식: 무작위 Train 80% / Test 20% 분할을 30회 반복하고 Accuracy·AUC를 평균한다.
- 우리 데이터 조건: 원본 448행, 13개 입력, 행마다 Unknown 총 4개, 서로 다른 마스킹 10세트를 유지한다.
- 같은 원본 입력과 그 변형이 양쪽에 섞이지 않도록 **그룹 단위 80:20**으로 나눈다. 그룹 크기가 달라 원본 행 비율은 매회 정확한 80:20이 아닐 수 있다.
- RF 세부 파라미터와 인코딩은 논문에 공개되지 않았다. 아래 scikit-learn 기본 설정과 원핫 인코딩은 **우리 기준 설정**이지 저자의 최적 파라미터가 아니다.
- 임계값은 0.5로 고정하고 Brier·Precision·Recall·F1·FP/FN도 함께 기록한다. GridSearch·앙상블·임계값 탐색은 이번 단계에서 실행하지 않는다.
- 기존 7:3 분할의 RF 튜닝·앙상블 노트북과 산출물은 변경하지 않는다. 이 노트북의 반복 평가 평균과 예전 단일 Test 점수를 같은 조건의 순위로 비교하지 않는다.
- scikit-learn RF는 MPS를 지원하지 않으므로 CPU를 사용한다.

출처: [Bohanec et al. (2017), 본문 4절·Table 3](https://doi.org/10.1016/j.eswa.2016.11.010),
[저자 공개 원문](https://www.researchgate.net/publication/309885039_Explaining_machine_learning_models_in_sales_predictions).
논문의 22개 입력·Accuracy 0.782·AUC 0.85를 그대로 재현한 실험은 아니다.


## 1. 라이브러리와 고정 설정


In [1]:
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from IPython import get_ipython
from IPython.display import display
from IPython.utils.capture import capture_output
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

SPLIT_RANDOM_STATE = 1
RF_RANDOM_STATE = 1
REPEAT_COUNT = 30
TEST_GROUP_FRACTION = 0.20
CLASSIFICATION_THRESHOLD = 0.5
REFERENCE_REPEAT = 1

## 2. 기존 전처리에서 전체 마스킹 데이터 가져오기

전처리 노트북만 실행한다. 원본 CSV 위치는 기존 `SALESLUV_B2B_DATA_PATH` 설정을 따른다.
전처리가 만드는 기존 7:3 Train/Test는 이번 평가에 사용하지 않고, 전체 원본과 마스킹 10세트에서 다시 나눈다.


In [2]:
current_dir = Path.cwd().resolve()
if current_dir.name == "notebooks":
    preprocessing_notebook = current_dir / "deal_data_preprocessing.ipynb"
elif current_dir.name == "backend" and (current_dir / "notebooks").is_dir():
    preprocessing_notebook = current_dir / "notebooks" / "deal_data_preprocessing.ipynb"
elif (current_dir / "backend" / "notebooks").is_dir():
    preprocessing_notebook = current_dir / "backend" / "notebooks" / "deal_data_preprocessing.ipynb"
else:
    raise RuntimeError("backend/notebooks 폴더를 찾을 수 없습니다.")

ipython = get_ipython()
assert ipython is not None, "Jupyter 커널에서 실행해야 합니다."
# 전처리에서 출력하는 원본 영업 행은 이 노트북 출력에 다시 저장하지 않는다.
with capture_output():
    ipython.run_line_magic("run", str(preprocessing_notebook))

MODEL_FEATURE_NAMES = ipython.user_ns["MODEL_FEATURE_NAMES"]
CATEGORY_VALUES = ipython.user_ns["CATEGORY_VALUES"]
X_categorical = ipython.user_ns["X_categorical"]
X_all_masked_sets = ipython.user_ns["X_all_masked_sets"]
y = ipython.user_ns["y"]
input_group_ids = ipython.user_ns["input_group_ids"]
SOURCE_SHA256 = ipython.user_ns["SOURCE_SHA256"]
UNKNOWN_COLUMNS_PER_ROW = ipython.user_ns["UNKNOWN_COLUMNS_PER_ROW"]

assert X_categorical.shape == (448, 13)
assert X_categorical.index.is_unique
assert X_categorical.index.equals(y.index)
assert y.index.equals(input_group_ids.index)
assert list(X_categorical.columns) == list(MODEL_FEATURE_NAMES)
assert set(y.unique()) == {0, 1}
assert len(X_all_masked_sets) == 10
assert UNKNOWN_COLUMNS_PER_ROW == 4
for masked_data in X_all_masked_sets.values():
    assert masked_data.index.equals(y.index)
    assert list(masked_data.columns) == list(MODEL_FEATURE_NAMES)
    assert masked_data.eq("Unknown").sum(axis=1).eq(UNKNOWN_COLUMNS_PER_ROW).all()
    assert (X_categorical.eq("Unknown") <= masked_data.eq("Unknown")).all().all()

# 각 마스킹 세트의 원본 행 순서와 정답 순서를 맞춘다.
X_all_raw = pd.concat(X_all_masked_sets, names=["mask_set", "original_row_id"])
y_all_masked = pd.concat(
    {set_name: y for set_name in X_all_masked_sets},
    names=["mask_set", "original_row_id"],
)
all_original_row_ids = X_all_raw.index.get_level_values("original_row_id")
assert X_all_raw.index.equals(y_all_masked.index)
assert X_all_raw.shape == (4480, 13)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 40)
pd.set_option("display.width", 180)
print(f"원본 {len(y)}행, 입력 {len(MODEL_FEATURE_NAMES)}개")
print(f"동일 입력 {input_group_ids.nunique()}그룹, 마스킹 {len(X_all_masked_sets)}세트")

원본 448행, 입력 13개
동일 입력 198그룹, 마스킹 10세트


### 해석

마스킹 4,480행은 새로운 영업 사례 4,480건이 아니라 원본 448건의 입력 변형이다.
13개 입력이 같은 원본 행들까지 한 그룹에 묶으므로 중복 행을 삭제하지 않으면서 평가 쪽으로 넘어가는 것을 막는다.
마스킹은 정답을 보지 않고 만든 기존 결과를 그대로 사용한다.


## 3. 튜닝하지 않은 RF 기준 설정


In [3]:
onehot = OneHotEncoder(
    categories=[list(CATEGORY_VALUES[column]) for column in MODEL_FEATURE_NAMES],
    drop="first",
    handle_unknown="error",
    sparse_output=False,
    dtype=np.float32,
)
rf_classifier = RandomForestClassifier(
    n_estimators=100,
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    class_weight=None,
    random_state=RF_RANDOM_STATE,
    n_jobs=1,
)
rf_template = Pipeline([("onehot", onehot), ("classifier", rf_classifier)])
display(
    pd.Series(
        rf_classifier.get_params(),
        name="우리 고정 설정",
    ).to_frame()
)

,우리 고정 설정
bootstrap,True
ccp_alpha,0.0
class_weight,None
criterion,gini
max_depth,None
max_features,sqrt
max_leaf_nodes,None
max_samples,None
min_impurity_decrease,0.0
min_samples_leaf,1


### 해석

나무 100개, 깊이 제한 없음, 잎의 최소 샘플 수 1을 출발점으로 고정했다. 최적 설정이라고 주장하지 않는다.
각 나무는 학습 행을 중복 추출하고, 분기마다 일부 피처를 무작위로 검토한다.
`max_features="sqrt"`는 원핫 변환 후 39개 피처를 기준으로 적용된다. Unknown도 정상 범주로 학습한다.

RF의 분기 기준은 Gini이며 Brier를 직접 최소화하며 학습하는 모델이 아니다.
이번 단계에서는 Brier를 평가 지표로만 계산한다. 이후 튜닝을 진행하면 Train 내부의 선택 기준으로 사용할 수 있다.


## 4. 그룹 단위 80:20 분할로 30회 학습·평가

매회 새 RF를 만들고 그 회차의 Train으로만 학습한다. Test는 같은 원본 행의 마스킹 10세트를 각각 평가한다.
RF 설정과 seed는 고정하고 분할만 바꾼다. 점수가 가장 좋은 회차를 고르거나 하나의 모델을 이어 학습하지 않는다.

[GroupShuffleSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html)의
`test_size=0.2`는 **그룹 수 기준**이다. 원본 Test 행 수·비율·Won 비율을 함께 저장한다.


In [4]:
def calculate_metrics(y_true, probability):
    """Won 확률을 0.5로 분류하고 확률·분류 성능을 함께 계산한다."""
    assert np.isfinite(probability).all()
    assert ((probability >= 0) & (probability <= 1)).all()
    prediction = (probability >= CLASSIFICATION_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, prediction),
        "auc": roc_auc_score(y_true, probability),
        "brier": brier_score_loss(y_true, probability),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
        "tp": int(tp),
        "fpr": float(fp / (fp + tn)),
    }


splitter = GroupShuffleSplit(
    n_splits=REPEAT_COUNT,
    test_size=TEST_GROUP_FRACTION,
    random_state=SPLIT_RANDOM_STATE,
)
evaluation_splits = list(splitter.split(X_categorical, y, groups=input_group_ids))
assert len({tuple(test) for _, test in evaluation_splits}) == REPEAT_COUNT

mask_result_rows = []
repeat_rows = []
started = perf_counter()

for repeat, (train_positions, test_positions) in enumerate(evaluation_splits, start=1):
    train_row_ids = y.index[train_positions]
    test_row_ids = y.index[test_positions]
    train_groups = input_group_ids.iloc[train_positions]
    test_groups = input_group_ids.iloc[test_positions]
    assert set(train_row_ids).isdisjoint(test_row_ids)
    assert set(train_row_ids) | set(test_row_ids) == set(y.index)
    assert set(train_groups).isdisjoint(test_groups)
    assert test_groups.nunique() == int(np.ceil(input_group_ids.nunique() * TEST_GROUP_FRACTION))
    assert set(y.iloc[train_positions].unique()) == {0, 1}
    assert set(y.iloc[test_positions].unique()) == {0, 1}

    # 원본 ID로 잘라 10개 변형 모두를 같은 Train에 넣는다.
    train_mask = all_original_row_ids.isin(train_row_ids)
    X_repeat_train = X_all_raw.loc[train_mask]
    y_repeat_train = y_all_masked.loc[train_mask]
    assert X_repeat_train.index.equals(y_repeat_train.index)
    assert len(X_repeat_train) == len(train_positions) * len(X_all_masked_sets)
    assert pd.Series(all_original_row_ids[train_mask]).value_counts().eq(10).all()

    fitted_rf = clone(rf_template)
    fitted_rf.fit(X_repeat_train, y_repeat_train)
    assert fitted_rf.named_steps["onehot"].transform(X_repeat_train.iloc[:1]).shape[1] == 39
    won_index = list(fitted_rf.classes_).index(1)
    mask_metrics = []
    for set_name, masked_data in X_all_masked_sets.items():
        X_repeat_test = masked_data.iloc[test_positions]
        y_repeat_test = y.iloc[test_positions]
        assert X_repeat_test.index.equals(y_repeat_test.index)
        probability = fitted_rf.predict_proba(X_repeat_test)[:, won_index]
        metrics = calculate_metrics(y_repeat_test, probability)
        mask_metrics.append(metrics)
        mask_result_rows.append({"repeat": repeat, "mask_set": set_name, **metrics})

    # 먼저 마스킹 10세트를 평균한다. 300세트를 서로 독립인 거래 표본으로 취급하지 않는다.
    mean_metrics = pd.DataFrame(mask_metrics).mean().to_dict()
    repeat_rows.append(
        {
            "repeat": repeat,
            "train_rows": len(train_positions),
            "test_rows": len(test_positions),
            "train_groups": train_groups.nunique(),
            "test_groups": test_groups.nunique(),
            "test_row_fraction": len(test_positions) / len(y),
            "test_won_fraction": float(y_repeat_test.mean()),
            **mean_metrics,
        }
    )
    # 저장할 회차는 실행 전에 1번으로 고정했다. 성능을 보고 좋은 seed를 고르지 않는다.
    if repeat == REFERENCE_REPEAT:
        model_rf_baseline = fitted_rf
        baseline_train_positions = train_positions.copy()
        baseline_test_positions = test_positions.copy()
    print(
        f"{repeat:02d}/{REPEAT_COUNT} 완료 | Train {len(train_positions)}, "
        f"Test {len(test_positions)} | Accuracy {mean_metrics['accuracy']:.4f}, "
        f"AUC {mean_metrics['auc']:.4f}, Brier {mean_metrics['brier']:.4f}"
    )

evaluation_seconds = perf_counter() - started
mask_results = pd.DataFrame(mask_result_rows)
repeat_results = pd.DataFrame(repeat_rows).set_index("repeat")
metric_names = list(mean_metrics)
assert len(repeat_results) == REPEAT_COUNT
assert len(mask_results) == REPEAT_COUNT * len(X_all_masked_sets)
assert mask_results.groupby("repeat")["mask_set"].nunique().eq(10).all()
assert repeat_results.notna().all().all()
np.testing.assert_allclose(
    mask_results.groupby("repeat")[metric_names].mean(),
    repeat_results[metric_names],
)
np.testing.assert_allclose(
    repeat_results[["fp", "fn", "tn", "tp"]].sum(axis=1),
    repeat_results["test_rows"],
)

01/30 완료 | Train 357, Test 91 | Accuracy 0.6846, AUC 0.7043, Brier 0.2314


02/30 완료 | Train 361, Test 87 | Accuracy 0.6598, AUC 0.7282, Brier 0.2180


03/30 완료 | Train 368, Test 80 | Accuracy 0.7238, AUC 0.7800, Brier 0.1970
04/30 완료 | Train 340, Test 108 | Accuracy 0.6787, AUC 0.7024, Brier 0.2243


05/30 완료 | Train 362, Test 86 | Accuracy 0.6302, AUC 0.6678, Brier 0.2493
06/30 완료 | Train 324, Test 124 | Accuracy 0.7048, AUC 0.7381, Brier 0.1958


07/30 완료 | Train 383, Test 65 | Accuracy 0.7600, AUC 0.8211, Brier 0.1712


...

24/30 완료 | Train 353, Test 95 | Accuracy 0.7305, AUC 0.7840, Brier 0.1919


25/30 완료 | Train 365, Test 83 | Accuracy 0.7012, AUC 0.7639, Brier 0.1989
26/30 완료 | Train 318, Test 130 | Accuracy 0.6700, AUC 0.7196, Brier 0.2139


27/30 완료 | Train 370, Test 78 | Accuracy 0.7077, AUC 0.7157, Brier 0.2208
28/30 완료 | Train 329, Test 119 | Accuracy 0.6126, AUC 0.6602, Brier 0.2393


29/30 완료 | Train 364, Test 84 | Accuracy 0.6679, AUC 0.7146, Brier 0.2248


30/30 완료 | Train 362, Test 86 | Accuracy 0.7186, AUC 0.7862, Brier 0.1942


### 해석

각 줄은 새로 나눈 Test에서 마스킹 10세트를 평가한 평균이다.
FP는 실제 Lost를 Won으로 예측해 주의할 거래를 놓치는 경우다. 반복마다 Test 건수가 달라 FP 건수와 함께
`fpr`(실제 Lost 중 잘못 Won으로 표시한 비율)도 본다.

기존 실험에서 이미 살펴본 데이터를 재사용하므로, 완전히 새로운 외부 검증 결과는 아니다.


## 5. 30회 평균과 분할별 결과


In [5]:
baseline_summary = repeat_results[metric_names].agg(["mean", "std"]).T
baseline_summary.columns = ["30회_평균", "반복간_표준편차"]
display(baseline_summary.round(6))
display(repeat_results.round(6))
print(f"30회 학습·평가 시간: {evaluation_seconds:.2f}초")
print(
    "원본 Test 행 비율: "
    f"최소 {repeat_results['test_row_fraction'].min():.1%}, "
    f"평균 {repeat_results['test_row_fraction'].mean():.1%}, "
    f"최대 {repeat_results['test_row_fraction'].max():.1%}"
)

,30회_평균,반복간_표준편차
accuracy,0.695847,0.039228
auc,0.740949,0.045089
brier,0.209706,0.019724
precision,0.695281,0.062083
recall,0.740104,0.064721
f1,0.714223,0.050065
fp,15.593333,4.740066
fn,12.543333,4.434480
tn,27.440000,3.498433
tp,36.256667,11.406824


,train_rows,test_rows,train_groups,test_groups,test_row_fraction,test_won_fraction,accuracy,auc,brier,precision,recall,f1,fp,fn,tn,tp,fpr
repeat,,,,,,,,,,,,,,,,,
1,357,91,158,40,0.203125,0.538462,0.684615,0.704300,0.231366,0.702312,0.718367,0.709853,14.9,13.8,27.1,35.2,0.354762
2,361,87,158,40,0.194196,0.551724,0.659770,0.728205,0.218044,0.718211,0.637500,0.672547,12.2,17.4,26.8,30.6,0.312821
3,368,80,158,40,0.178571,0.437500,0.723750,0.779968,0.196992,0.640379,0.842857,0.727459,16.6,5.5,28.4,29.5,0.368889
4,340,108,158,40,0.241071,0.546296,0.678704,0.702352,0.224343,0.681329,0.774576,0.724511,21.4,13.3,27.6,45.7,0.436735
5,362,86,158,40,0.191964,0.523256,0.630233,0.667778,0.249253,0.626424,0.728889,0.673652,19.6,12.2,21.4,32.8,0.478049
6,324,124,158,40,0.276786,0.653226,0.704839,0.738099,0.195783,0.772737,0.776543,0.773960,18.5,18.1,24.5,62.9,0.430233
7,383,65,158,40,0.145089,0.400000,0.760000,0.821055,0.171225,0.672610,0.780769,0.722127,9.9,5.7,29.1,20.3,0.253846
8,369,79,158,40,0.176339,0.594937,0.735443,0.807314,0.192784,0.846966,0.676596,0.751026,5.7,15.2,26.3,31.8,0.178125
9,389,59,158,40,0.131696,0.406780,0.623729,0.660595,0.240408,0.537237,0.537500,0.534503,11.1,11.1,23.9,12.9,0.317143


30회 학습·평가 시간: 6.19초
원본 Test 행 비율: 최소 13.2%, 평균 20.5%, 최대 29.0%


### 해석

기준 성능은 30회 점수의 평균이다. 표준편차는 분할을 바꿨을 때 점수가 얼마나 흔들렸는지 보여준다.
반복 사이에 같은 거래가 다시 등장하므로 이 표준편차를 독립 표본의 신뢰구간으로 해석하지 않는다.

Accuracy는 맞힌 비율, AUC는 Won을 Lost보다 높은 확률로 구분하는 능력이다.
Brier는 예측 확률과 실제 정답의 차이를 보며 작을수록 좋다. FP/FN은 평균 건수이므로 소수로 표시된다.
논문과 달리 13개 컬럼·추가 Unknown·그룹 분리를 사용하므로 논문 수치와 직접적인 우열을 단정하지 않는다.


## 6. 기준 모델 저장과 재로드 확인

결과를 보고 고른 모델이 아니라 **사전에 지정한 첫 번째 분할의 Train 학습 모델**을 저장한다.
30회 평균은 평가 절차 전체의 성적이고, 저장된 모델 하나의 독립 Test 성적이 아니다.
후속 튜닝·앙상블은 아래 분할과 RF 설정을 재사용해 같은 조건으로 비교해야 한다.


In [6]:
# 원본 영업 행 대신 허용 범주로 만든 합성 입력을 저장·로드 검증에 사용한다.
known_row = {
    column: next(value for value in CATEGORY_VALUES[column] if value != "Unknown")
    for column in MODEL_FEATURE_NAMES
}
masked_row = {**known_row, **dict.fromkeys(MODEL_FEATURE_NAMES[:4], "Unknown")}
self_check_X = pd.DataFrame(
    [known_row, masked_row, dict.fromkeys(MODEL_FEATURE_NAMES, "Unknown")],
    columns=list(MODEL_FEATURE_NAMES),
)
expected_probability = model_rf_baseline.predict_proba(self_check_X)
assert np.isfinite(expected_probability).all()
np.testing.assert_allclose(expected_probability.sum(axis=1), 1.0)

artifact_path = (
    preprocessing_notebook.parents[1]
    / "pipeline"
    / "artifacts"
    / "deal-paper-rf-baseline-v1.joblib"
)
artifact_path.parent.mkdir(parents=True, exist_ok=True)
bundle = {
    "schema_version": 1,
    "model_version": "deal-paper-rf-baseline-v1",
    "model": model_rf_baseline,
    "model_feature_names": list(MODEL_FEATURE_NAMES),
    "category_values": {column: list(values) for column, values in CATEGORY_VALUES.items()},
    "target": {"Lost": 0, "Won": 1},
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "source_sha256": SOURCE_SHA256,
    "paper_doi": "10.1016/j.eswa.2016.11.010",
    "exact_paper_reproduction": False,
    "rf_params": rf_classifier.get_params(),
    "training_scope": "reference_split_train_only",
    "reference_repeat": REFERENCE_REPEAT,
    "training_original_rows": len(baseline_train_positions),
    "training_masked_rows": len(baseline_train_positions) * len(X_all_masked_sets),
    "reference_train_positions": baseline_train_positions,
    "reference_test_positions": baseline_test_positions,
    "reference_test_metrics": repeat_results.loc[REFERENCE_REPEAT, metric_names].to_dict(),
    "evaluation": {
        "splitter": "GroupShuffleSplit",
        "split_random_state": SPLIT_RANDOM_STATE,
        "test_group_fraction": TEST_GROUP_FRACTION,
        "repeat_count": REPEAT_COUNT,
        "masking_set_count": len(X_all_masked_sets),
        "unknown_columns_per_row": UNKNOWN_COLUMNS_PER_ROW,
        "splits": evaluation_splits,
        "repeat_results": repeat_results,
        "mask_results": mask_results,
        "summary": baseline_summary,
        "seconds": evaluation_seconds,
    },
    "versions": {name: version(name) for name in ("scikit-learn", "numpy", "pandas", "joblib")},
}
joblib.dump(bundle, artifact_path)
restored = joblib.load(artifact_path)
np.testing.assert_allclose(
    expected_probability,
    restored["model"].predict_proba(self_check_X),
    rtol=1e-12,
    atol=1e-12,
)
assert set(restored["model"].classes_) == {0, 1}
assert len(restored["model_feature_names"]) == 13
assert restored["evaluation"]["repeat_count"] == 30
assert set(restored["reference_train_positions"]).isdisjoint(restored["reference_test_positions"])
print(f"기준 모델 저장: backend/pipeline/artifacts/{artifact_path.name}")
print(f"저장 모델의 학습 범위: 1회차 Train 원본 {len(baseline_train_positions)}건")
print(f"파일 크기: {artifact_path.stat().st_size / 1024**2:.3f} MiB")
print("13개 입력·마스킹·30회 그룹 분리·저장 후 재로드 검증: 통과")

기준 모델 저장: backend/pipeline/artifacts/deal-paper-rf-baseline-v1.joblib
저장 모델의 학습 범위: 1회차 Train 원본 357건
파일 크기: 16.158 MiB
13개 입력·마스킹·30회 그룹 분리·저장 후 재로드 검증: 통과


### 해석

원핫 인코더·RF·입력 계약·평가 분할·집계 결과를 joblib 파일 하나에 저장했다.
기존 튜닝 RF와 Stacking 파일은 덮어쓰지 않았고, 백엔드나 AWS 모델도 교체하지 않는다.
저장된 기준 모델에 Test를 추가 학습하지 않았다. 추후 전체 데이터로 다시 학습하면 별도 모델로 구분해야 한다.
원본 영업 행은 이 노트북 출력에 포함하지 않으며 모델 산출물은 Git에 올리지 않는다.
